## Section 0 — Setup & Environment Detection

In [ ]:
# Section 0 — Setup & Environment Detection
import os, sys, importlib.util

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_KAGGLE:
    OUTPUT_DIR = '/kaggle/working/'
    print('Environment : Kaggle')
else:
    OUTPUT_DIR = '/content/output/'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print('Environment : Colab (or local)')

print(f'Output dir  : {OUTPUT_DIR}')

import subprocess

def _pip(*pkgs):
    """Install packages only if they are not already importable."""
    _name_map = {'scikit-learn': 'sklearn', 'vaderSentiment': 'vaderSentiment'}
    for pkg in pkgs:
        mod = _name_map.get(pkg, pkg.replace('-', '_').split('[')[0])
        if importlib.util.find_spec(mod) is None:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

_pip('vaderSentiment', 'kagglehub', 'wordcloud', 'plotly', 'scikit-learn', 'textblob', 'statsmodels')
print('Dependencies OK')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy import stats
from wordcloud import WordCloud, STOPWORDS
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from datetime import datetime
import re, warnings
warnings.filterwarnings('ignore')

vader = SentimentIntensityAnalyzer()

# ── Constants ──────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SEG_COLORS = {
    'Champions':    '#27ae60',
    'Rising Stars': '#2980b9',
    'Core':         '#7f8c8d',
    'Vulnerable':   '#c0392b',
}
MIN_CAT_N = 20

def section_header(title):
    """Print a consistent section banner to the console."""
    width = max(len(title) + 6, 54)
    bar = '\u2550' * width
    print(f'\n{bar}')
    print(f'  {title}')
    print(f'{bar}')

def savefig(filename):
    """Save the current matplotlib figure to OUTPUT_DIR at 150 dpi."""
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  \u2192 {path}')

section_header('AMAZON PRODUCT & CUSTOMER BEHAVIOR ANALYZER')


## Section 1 — Data Loading

In [ ]:
# Section 1 — Data Loading
import kagglehub, glob

path = kagglehub.dataset_download('karkavelrajaj/amazon-sales-dataset')
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
print('Dataset path :', path)
print('CSV files    :', csv_files)

df_raw = pd.read_csv(csv_files[0])
print(f'\nRaw shape    : {df_raw.shape}')
print('\nColumns:')
print(df_raw.columns.tolist())
df_raw.head(3)


## Section 2 — Clean & Detect Columns

In [ ]:
# Section 2 — Clean & Detect Columns

df = df_raw.copy()

# ── helper: first column whose name matches any keyword ──────────────────────
def detect_col(df, *keywords):
    """Return the first column name matching any of the given keywords (case-insensitive)."""
    for kw in keywords:
        for c in df.columns:
            if kw.lower() in c.lower():
                return c
    return None

COL_PRODUCT   = detect_col(df, 'product_id', 'product id', 'asin')
COL_NAME      = detect_col(df, 'product_name', 'name', 'title')
COL_CAT       = detect_col(df, 'category')
COL_DISC_PCT  = detect_col(df, 'discount_percentage', 'discount_pct', 'discount%')
COL_DISC_P    = detect_col(df, 'discounted_price', 'sale_price')
COL_ACT_P     = detect_col(df, 'actual_price', 'original_price', 'mrp')
COL_RATING    = detect_col(df, 'rating')
COL_RCOUNT    = detect_col(df, 'rating_count', 'review_count', 'num_ratings')
COL_REVIEW    = detect_col(df, 'review_content', 'review_text', 'review_body')
COL_RTITLE    = detect_col(df, 'review_title')

# ── Column summary ────────────────────────────────────────────────────────────
COLUMN_SUMMARY = {
    'product_id':     COL_PRODUCT,
    'name':           COL_NAME,
    'category':       COL_CAT,
    'discount_pct':   COL_DISC_PCT,
    'actual_price':   COL_ACT_P,
    'discounted_price': COL_DISC_P,
    'rating':         COL_RATING,
    'rating_count':   COL_RCOUNT,
    'review':         COL_REVIEW,
    'review_title':   COL_RTITLE,
}

section_header('Section 2 — Column Detection')
print(f'  {"Field":<20} {"Mapped column"}')
print('  ' + '─' * 45)
for k, v in COLUMN_SUMMARY.items():
    status = str(v) if v else '⚠️  NOT FOUND'
    print(f'  {k:<20} {status}')

if COL_RATING is None:
    print('\n⚠️  No rating column found — pressure/traction signals will use neutral fallback.')

# ── Data quality report ──────────────────────────────────────────────────────
key_cols = [c for c in [COL_PRODUCT, COL_NAME, COL_CAT, COL_DISC_PCT,
                         COL_ACT_P, COL_RATING, COL_RCOUNT] if c]
print('\n── Data Quality Report ──────────────────────────────')
print(f'{"Column":<30} {"Missing %":>10}')
for c in key_cols:
    pct = df[c].isna().mean() * 100
    print(f'  {c:<28} {pct:>9.1f}%')

rows_before = len(df)
drop_subset = [c for c in [COL_PRODUCT, COL_CAT, COL_RATING] if c is not None]
if drop_subset:
    df.dropna(subset=drop_subset, inplace=True)
rows_dropped = rows_before - len(df)
print(f'\nTotal rows dropped (missing ID/Category/Rating): {rows_dropped}')
print(f'Rows remaining: {len(df)}')

# ── Type conversions ──────────────────────────────────────────────────────────
def clean_price(s):
    """Parse a price string (possibly with currency symbols/commas) to float."""
    if pd.isna(s):
        return np.nan
    cleaned = re.sub(r'[^\d.]', '', str(s).replace(',', ''))
    try:
        return float(cleaned)
    except (ValueError, TypeError):
        return np.nan

def clean_pct(s):
    """Parse a percentage string (e.g. '42%') to a 0–1 float."""
    if pd.isna(s):
        return np.nan
    cleaned = re.sub(r'[^\d.]', '', str(s))
    try:
        return float(cleaned) / 100
    except (ValueError, TypeError):
        return np.nan

def clean_rating(s):
    """Parse a rating string (e.g. '4.2 out of 5') to float."""
    if pd.isna(s):
        return np.nan
    try:
        return float(str(s).split()[0])
    except:
        return np.nan

def clean_count(s):
    """Parse a review/rating count string (e.g. '1,234') to float."""
    if pd.isna(s):
        return np.nan
    cleaned = re.sub(r'[^\d]', '', str(s))
    try:
        return float(cleaned)
    except (ValueError, TypeError):
        return np.nan

df['discount_pct']     = df[COL_DISC_PCT].apply(clean_pct) if COL_DISC_PCT is not None else np.nan
df['actual_price']     = df[COL_ACT_P].apply(clean_price) if COL_ACT_P is not None else np.nan
df['discounted_price'] = df[COL_DISC_P].apply(clean_price) if COL_DISC_P is not None else np.nan
df['rating']           = df[COL_RATING].apply(clean_rating) if COL_RATING is not None else 3.0
df['review_count']     = df[COL_RCOUNT].apply(clean_count) if COL_RCOUNT is not None else 1.0

# Derive discount_pct from prices if not directly available
if df['discount_pct'].isna().mean() > 0.5:
    df['discount_pct'] = 1 - df['discounted_price'] / df['actual_price']
df['discount_pct'] = df['discount_pct'].clip(0, 1)

# Fill medians
for col in ['discount_pct', 'actual_price', 'discounted_price', 'rating', 'review_count']:
    med = df[col].median()
    df[col].fillna(med, inplace=True)

# ── Deduplicate on product ID ──────────────────────────────────────────────────
if COL_PRODUCT is not None:
    df = df.drop_duplicates(subset=[COL_PRODUCT])

print(f'\nFinal deduped shape : {df.shape}')

# ── Category cleanup ──────────────────────────────────────────────────────────
if COL_CAT is not None:
    df['category'] = df[COL_CAT].astype(str).str.split('|').str[0].str.strip()
    df['category'] = df['category'].str.replace(r'&amp;', '&', regex=False)
    df['category'] = df['category'].str.replace(r'\s+', ' ', regex=True)
else:
    df['category'] = 'Unknown'

cat_counts = df['category'].value_counts()
valid_cats = cat_counts[cat_counts >= MIN_CAT_N].index
df = df[df['category'].isin(valid_cats)].copy()

print(f'\nProducts after MIN_CAT_N={MIN_CAT_N} filter : {len(df):,}')
print(f'Categories : {df["category"].nunique()}')
print('\nCategory distribution:')
print(df['category'].value_counts().to_string())


## Section 3 — Feature Engineering

In [ ]:
# Section 3 — Feature Engineering

# ── Engagement proxy ──────────────────────────────────────────────────────────
df['log_review_count'] = np.log1p(df['review_count'])
rc_max = df['log_review_count'].max() or 1
df['engagement'] = df['log_review_count'] / rc_max   # 0–1

# ── Rating normalised ─────────────────────────────────────────────────────────
df['rating_norm'] = (df['rating'] - df['rating'].min()) / \
                    (df['rating'].max() - df['rating'].min() + 1e-9)

# ── Category-level aggregates ─────────────────────────────────────────────────
cat_stats = df.groupby('category').agg(
    cat_avg_rating=('rating', 'mean'),
    cat_avg_disc=('discount_pct', 'mean'),
    cat_count=('rating', 'count')
).reset_index()
df = df.merge(cat_stats, on='category', how='left')

# Normalise category-level rating
cr_min, cr_max = df['cat_avg_rating'].min(), df['cat_avg_rating'].max()
df['cat_rating_norm'] = (df['cat_avg_rating'] - cr_min) / (cr_max - cr_min + 1e-9)

print(f'Feature engineering done. Shape: {df.shape}')
print(df[['rating', 'discount_pct', 'engagement', 'rating_norm', 'cat_rating_norm']].describe().round(3).to_string())


## Section 4 — Sentiment Analysis

In [ ]:
# Section 4 — Sentiment Analysis

# Minimum number of products with usable text for NLP-based sentiment
MIN_TEXT_PRODUCTS = 100

text_col = COL_REVIEW or COL_RTITLE or COL_NAME
has_text  = df[text_col].notna() & (df[text_col].astype(str).str.strip() != '') \
            if text_col else pd.Series(False, index=df.index)

if has_text.sum() <= MIN_TEXT_PRODUCTS:
    print(f'⚠  Only {has_text.sum()} rows have usable review text in column '
          f'"{text_col}".')
    print('   → Using rating-based proxy for sentiment: '
          'sentiment = (rating − 3) / 2')
    print('   This proxy is applied to ALL rows to ensure consistent coverage.')
    df['sentiment_score'] = (df['rating'] - 3) / 2
    df['sentiment_raw']   = df['sentiment_score']
else:
    print(f'Running VADER + TextBlob ensemble on {has_text.sum()} texts …')

    def sentiment_ensemble(txt):
        """Blend VADER and TextBlob polarity scores (equal weight) into a single value."""
        if not isinstance(txt, str) or not txt.strip():
            return np.nan
        v = vader.polarity_scores(txt)['compound']          # −1 … +1
        t = TextBlob(txt).sentiment.polarity               # −1 … +1
        return (v + t) / 2

    df.loc[has_text, 'sentiment_raw'] = \
        df.loc[has_text, text_col].apply(sentiment_ensemble)
    df['sentiment_raw'].fillna((df['rating'] - 3) / 2, inplace=True)

    # Clip to avoid extreme outliers inflating std before z-scoring
    df['sentiment_raw'] = df['sentiment_raw'].clip(-1, 1)

    # Z-score → back to −1…+1 range via sigmoid-like scale
    mu, sigma = df['sentiment_raw'].mean(), df['sentiment_raw'].std() + 1e-9
    df['sentiment_score'] = ((df['sentiment_raw'] - mu) / sigma).clip(-3, 3) / 3

# Normalise to 0–1
s_min, s_max = df['sentiment_score'].min(), df['sentiment_score'].max()
df['sentiment_norm'] = (df['sentiment_score'] - s_min) / (s_max - s_min + 1e-9)

print(f'\nSentiment score  mean={df["sentiment_norm"].mean():.3f}  '
      f'std={df["sentiment_norm"].std():.3f}')

# ── Calibration histogram: raw vs calibrated scores ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
if 'sentiment_raw' in df.columns and df['sentiment_raw'].nunique() > 2:
    ax.hist(df['sentiment_raw'], bins=40, alpha=0.55, label='Raw sentiment',
            color='steelblue', edgecolor='white')
ax.hist(df['sentiment_score'], bins=40, alpha=0.55, label='Calibrated (Z-scored)',
        color='coral', edgecolor='white')
ax.axvline(df['sentiment_score'].mean(), color='darkred', linestyle='--',
           linewidth=1.5, label=f'Mean={df["sentiment_score"].mean():.3f}')
ax.set_title('Sentiment Calibration — Raw vs Z-Scored', fontsize=12, fontweight='bold')
ax.set_xlabel('Sentiment Score')
ax.set_ylabel('Count')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## Section 5 — Pressure / Traction / PVS Indices

In [ ]:
# Section 5 — Pressure / Traction / PVS Indices

# ── Weight constants (easy to tune) ──────────────────────────────────────────
W_PRESSURE = dict(disc=0.40, rating_risk=0.40, low_eng=0.20)
W_TRACTION = dict(eng=0.35, cat_rating=0.35, sentiment=0.30)
W_PVS      = dict(traction=0.50, cat_rating=0.30, sentiment=0.20)

# ── Pressure Index (all components are percentile ranks) ─────────────────────
# High discount, low rating, low engagement → high pressure on the product
disc_rank        = df['discount_pct'].rank(pct=True)
rating_risk_rank = df['rating_norm'].rank(pct=True, ascending=False)   # low rating → high risk
low_eng_rank     = df['engagement'].rank(pct=True, ascending=False)    # low engagement → more pressure

df['Pressure'] = (W_PRESSURE['disc']        * disc_rank +
                  W_PRESSURE['rating_risk'] * rating_risk_rank +
                  W_PRESSURE['low_eng']     * low_eng_rank)

# ── Traction Index (all components are percentile ranks) ─────────────────────
eng_rank     = df['engagement'].rank(pct=True)
cat_rat_rank = df['cat_rating_norm'].rank(pct=True)
sent_rank    = df['sentiment_norm'].rank(pct=True)

df['Traction'] = (W_TRACTION['eng']        * eng_rank +
                  W_TRACTION['cat_rating'] * cat_rat_rank +
                  W_TRACTION['sentiment']  * sent_rank)

# ── Perceived Value Score (percentile ranks) ──────────────────────────────────
traction_rank = df['Traction'].rank(pct=True)
cat_rat_rank2 = df['cat_rating_norm'].rank(pct=True)
sent_rank2    = df['sentiment_norm'].rank(pct=True)

df['PVS'] = (W_PVS['traction']   * traction_rank +
             W_PVS['cat_rating'] * cat_rat_rank2 +
             W_PVS['sentiment']  * sent_rank2)

# ── Churn_Risk: percentile-calibrated ─────────────────────────────────────────
p75 = df['Pressure'].quantile(0.75)
p50 = df['Pressure'].quantile(0.50)
df['Churn_Risk'] = pd.cut(df['Pressure'],
                           bins=[-float('inf'), p50, p75, float('inf')],
                           labels=['Low', 'Medium', 'High'])

print(f'Pressure  Mean={df["Pressure"].mean():.3f}  Std={df["Pressure"].std():.3f}')
print(f'Traction  Mean={df["Traction"].mean():.3f}  Std={df["Traction"].std():.3f}')
print(f'PVS       Mean={df["PVS"].mean():.3f}  Std={df["PVS"].std():.3f}')

churn_counts = df['Churn_Risk'].value_counts()
print(f"\nChurn Risk  High={churn_counts.get('High',0)}  "
      f"Medium={churn_counts.get('Medium',0)}  Low={churn_counts.get('Low',0)}")

# ── Pearson Correlation Table ─────────────────────────────────────────────────
score_corr = df[['Pressure', 'Traction', 'PVS']].corr()
print('\n── Pearson Correlation Table (Pressure · Traction · PVS) ────')
print(score_corr.round(3).to_string())
pt_corr = score_corr.loc['Pressure', 'Traction']
if abs(pt_corr) > 0.6:
    print(f'\n⚠️  |corr(Pressure, Traction)| = {abs(pt_corr):.3f} > 0.6')
    print('   The two axes may not be independent enough — see model health note below.')

# ── Distribution plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes,
                           ['Pressure', 'Traction', 'PVS'],
                           ['#e74c3c', '#3498db', '#2ecc71']):
    ax.hist(df[col], bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Mean={df[col].mean():.3f}')
    ax.set_title(col, fontsize=14, fontweight='bold')
    ax.set_xlabel('Score (0–1)')
    ax.set_ylabel('Count')
    ax.legend()
plt.suptitle('Distribution of Pressure / Traction / PVS Indices', fontsize=16, y=1.02)
plt.tight_layout()
savefig('score_distributions.png')
plt.show()


### Model Health: Pressure–Traction Correlation

A key design requirement of the two-axis model is that **Pressure** and **Traction** measure
different things. If their Pearson correlation exceeds |0.6|, it suggests the axes are not
independent — products that face high pressure also tend to have high (or low) traction —
which undermines the four-quadrant logic.

Typical causes: shared raw signals (e.g. both axes incorporate rating or engagement),
or a dataset where discount and quality always move together. Review the weight constants
`W_PRESSURE` and `W_TRACTION` in Section 5 to reduce signal overlap.

## Section 6 — Quadrant Segmentation

In [ ]:
# Section 6 — Quadrant Segmentation

LOW_CONFIDENCE_THRESHOLD = 0.30

# ── Median-split quadrant assignment ─────────────────────────────────────────
pressure_med = df['Pressure'].median()
traction_med = df['Traction'].median()

def _quadrant_label(row):
    """Assign segment based on Pressure × Traction quadrant (median split)."""
    hi_traction = row['Traction'] >= traction_med
    hi_pressure = row['Pressure'] >= pressure_med
    if hi_traction and not hi_pressure:
        return 'Champions'       # hi traction · lo pressure
    elif hi_traction and hi_pressure:
        return 'Rising Stars'    # hi traction · hi pressure
    elif not hi_traction and not hi_pressure:
        return 'Core'            # lo traction · lo pressure
    else:
        return 'Vulnerable'      # lo traction · hi pressure

df['Segment'] = df.apply(_quadrant_label, axis=1)

# ── K-Means (k=4) — used ONLY for Confidence score ───────────────────────────
features = ['Pressure', 'Traction', 'PVS', 'discount_pct', 'rating_norm', 'engagement']
X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=20, max_iter=500)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Confidence: distance-based score (how cleanly a product sits in its quadrant)
distances = kmeans.transform(X_scaled)
nearest   = distances.min(axis=1)
second    = np.sort(distances, axis=1)[:, 1]
df['confidence'] = 1 - nearest / (nearest + second + 1e-9)

# ── Print segment summary ──────────────────────────────────────────────────────
total = len(df)
print(f'\n{"Segment":<15} {"Count":>7}  {"Share %":>8}  {"Low-conf (<{:.0f}%)".format(LOW_CONFIDENCE_THRESHOLD*100):>14}')
print('─' * 52)
for seg in ['Champions', 'Rising Stars', 'Core', 'Vulnerable']:
    mask = df['Segment'] == seg
    n    = mask.sum()
    pct  = n / total * 100
    low  = (mask & (df['confidence'] < LOW_CONFIDENCE_THRESHOLD)).sum()
    print(f'  {seg:<13} {n:>7}  {pct:>7.1f}%  {low:>14}')
print('─' * 52)
print(f'  {"TOTAL":<13} {total:>7}  {"100.0%":>8}')
print(f'\nMean confidence: {df["confidence"].mean():.3f}')
print(f'Pressure median: {pressure_med:.3f}')
print(f'Traction median: {traction_med:.3f}')

# ── SEGMENT_STATS — mean/std/count per segment ────────────────────────────────
SEGMENT_STATS = df.groupby('Segment')[['PVS', 'Pressure', 'Traction', 'confidence']].agg(
    ['mean', 'std', 'count'])
print('\n── SEGMENT_STATS ────────────────────────────────────────────')
print(SEGMENT_STATS.round(3).to_string())

# ── Quadrant scatter plot (Pressure × Traction) ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
for seg, color in SEG_COLORS.items():
    mask = df['Segment'] == seg
    ax.scatter(df.loc[mask, 'Pressure'], df.loc[mask, 'Traction'],
               c=color, label=f'{seg} ({mask.sum()})',
               alpha=0.6, s=df.loc[mask, 'confidence'] * 60 + 10)
ax.axvline(pressure_med, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Pressure median')
ax.axhline(traction_med, color='black', linestyle=':',  linewidth=1, alpha=0.5, label='Traction median')
ax.set_xlabel('Pressure Index', fontsize=12)
ax.set_ylabel('Traction Index', fontsize=12)
ax.set_title('Quadrant Map — Pressure × Traction', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
savefig('quadrant_map.png')
plt.show()


In [ ]:
# Segment symmetry note (median-split guarantees near-equal Champions/Vulnerable)
seg_counts = df['Segment'].value_counts()
champ_pct = seg_counts.get('Champions', 0) / len(df)
vuln_pct  = seg_counts.get('Vulnerable', 0) / len(df)
print(f"Champions  : {champ_pct:.1%}  |  Vulnerable: {vuln_pct:.1%}")
print("Note: median-split guarantees Champions ≈ Vulnerable in size (both span 50% of each axis).")


## Section 7 — Suspicious Review Detection

In [ ]:
# Section 7 — Suspicious Review Detection

# Number of rule-based signals a product must trigger to be flagged as suspicious
SUSPICION_THRESHOLD = 1

iso_features = ['review_count', 'discount_pct', 'rating', 'engagement']
X_iso = df[iso_features].fillna(df[iso_features].median()).values

iso_forest = IsolationForest(contamination=0.07, random_state=RANDOM_STATE)
df['anomaly_score'] = iso_forest.fit_predict(X_iso)          # −1 = anomaly
df['iso_score_raw'] = iso_forest.decision_function(X_iso)    # lower = more anomalous

df['suspicious'] = df['anomaly_score'] == -1
sus_count = df['suspicious'].sum()
sus_pct   = sus_count / len(df) * 100
print(f'Suspicious reviews : {sus_count} ({sus_pct:.1f}%)')

# ── Risk level classification ─────────────────────────────────────────────────
score_pcts = df['iso_score_raw'].quantile([0.10, 0.25])
def risk_level(row):
    """Classify each product's review risk as High, Medium, or Low."""
    if row['suspicious']:
        if row['iso_score_raw'] <= score_pcts[0.10]:
            return 'High'
        return 'Medium'
    return 'Low'
df['Risk_Level'] = df.apply(risk_level, axis=1)
print('\nRisk level distribution:')
print(df['Risk_Level'].value_counts().to_string())

# ── Summary DataFrame per Risk_Level ─────────────────────────────────────────
risk_summary = df.groupby('Risk_Level').agg(
    Count=('rating', 'count'),
    Mean_Rating=('rating', 'mean'),
    Mean_ReviewCount=('review_count', 'mean'),
    Mean_Discount=('discount_pct', 'mean'),
).round(3)
risk_summary['Mean_Discount'] = (risk_summary['Mean_Discount'] * 100).round(1)
risk_summary.rename(columns={'Mean_Discount': 'Mean_Discount_%'}, inplace=True)
print('\n── Risk-Level Summary (why products were flagged) ────')
print(risk_summary.to_string())

# ── Save suspicious products ──────────────────────────────────────────────────
sus_cols = [c for c in [COL_PRODUCT, COL_NAME, 'category', 'rating', 'discount_pct',
                         'review_count', 'Risk_Level', 'iso_score_raw'] if c in df.columns]
suspicious_path = os.path.join(OUTPUT_DIR, 'suspicious_products.csv')
df[df['suspicious']][sus_cols].to_csv(suspicious_path, index=False)
print(f'\nSaved suspicious_products.csv ({df["suspicious"].sum()} rows) → {suspicious_path}')

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_risk = {'Low': '#27ae60', 'Medium': '#e67e22', 'High': '#c0392b'}

for ax, x_col, xlabel in zip(axes,
                              ['discount_pct', 'rating'],
                              ['Discount %', 'Rating']):
    for level, color in colors_risk.items():
        mask = df['Risk_Level'] == level
        ax.scatter(df.loc[mask, x_col], df.loc[mask, 'review_count'],
                   c=color, label=level, alpha=0.5, s=25)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Review Count')
    ax.set_title(f'Risk Level by {xlabel}')
    ax.legend()
plt.suptitle('Suspicious Review Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('03_suspicious_reviews.png')
plt.show()


## Section 8 — RFM Analysis

In [ ]:
# Section 8 — RFM Analysis

section_header('Section 8 — RFM Analysis')
print("  R (Recency)   → inverse discount dependency (low discount = less price-driven)")
print("  F (Frequency) → review count (log-normalised) — interaction volume proxy")
print("  M (Monetary)  → actual_price × rating_norm — perceived revenue contribution")
print()

# ── Quintile helper with qcut → cut fallback ──────────────────────────────────
def quintile(series, n=5):
    """Assign quintile ranks 1–n, falling back to equal-width bins if qcut fails."""
    try:
        return pd.qcut(series.rank(method='first'), q=n,
                       labels=range(1, n + 1)).astype(int)
    except ValueError:
        return pd.cut(series, bins=n, labels=range(1, n + 1),
                      include_lowest=True).astype(int)

# ── Compute R, F, M proxies ────────────────────────────────────────────────────
df['R_score'] = quintile(df['review_count'])
df['F_score'] = quintile(df['log_review_count'])
# Monetary proxy
df['monetary_proxy'] = df['actual_price'] * df['rating_norm']
df['M_score'] = quintile(df['monetary_proxy'])
df['RFM_score'] = df['R_score'] + df['F_score'] + df['M_score']

# ── RFM segment labels ─────────────────────────────────────────────────────────
def rfm_segment(score):
    """Map a total RFM score to a named segment tier."""
    if score >= 13:
        return 'Champions'
    elif score >= 10:
        return 'Loyal'
    elif score >= 7:
        return 'Potential'
    elif score >= 5:
        return 'At Risk'
    else:
        return 'Inactive'

df['RFM_Segment'] = df['RFM_score'].apply(rfm_segment)
print('RFM segment distribution:')
print(df['RFM_Segment'].value_counts().to_string())

# ── Bar chart ─────────────────────────────────────────────────────────────────
rfm_order = ['Champions', 'Loyal', 'Potential', 'At Risk', 'Inactive']
rfm_counts = df['RFM_Segment'].value_counts().reindex(rfm_order, fill_value=0)
rfm_colors = ['#27ae60', '#2980b9', '#f39c12', '#e74c3c', '#7f8c8d']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(rfm_counts.index, rfm_counts.values, color=rfm_colors, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=11)
ax.set_xlabel('RFM Segment', fontsize=12)
ax.set_ylabel('Number of Products', fontsize=12)
ax.set_title('Product Distribution by RFM Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('rfm.png')
plt.show()


## Section 9 — Sensitivity & Validation

In [ ]:
# Section 9 — Sensitivity & Validation

# ── Silhouette Score ──────────────────────────────────────────────────────────
sil_score = silhouette_score(X_scaled, df['cluster'], random_state=RANDOM_STATE)
print(f'Silhouette Score  : {sil_score:.4f}')

if sil_score < 0.20:
    print(f"⚠️  Silhouette={sil_score:.4f} — quadrant boundaries are weak.")
    print("    Consider: the dataset may not have 4 natural clusters.")
    print("    Tip: Quadrant segmentation is still valid by design.")

# Per-sample silhouette
sil_samples = silhouette_samples(X_scaled, df['cluster'])
df['silhouette'] = sil_samples

# ── Per-segment silhouette means ──────────────────────────────────────────────
print('\n── Per-Segment Silhouette Means ─────────────────────────────')
print(f'  {"Segment":<15} {"Mean Silhouette":>16} {"Count":>7}')
print('  ' + '─' * 42)
for seg in ['Champions', 'Rising Stars', 'Core', 'Vulnerable']:
    mask = df['Segment'] == seg
    seg_sil = sil_samples[mask.values]
    if len(seg_sil) > 0:
        print(f'  {seg:<15} {seg_sil.mean():>16.4f} {len(seg_sil):>7}')

# ── ANOVA — does PVS differ significantly across segments? ────────────────────
groups = [df.loc[df['Segment'] == s, 'PVS'].values
          for s in ['Champions', 'Rising Stars', 'Core', 'Vulnerable']]
f_stat, p_val = stats.f_oneway(*groups)
print(f'\nANOVA F={f_stat:.2f}  p={p_val:.2e}')

# ── Tukey HSD pairwise tests ──────────────────────────────────────────────────
pvs_vals  = df['PVS'].values
seg_vals  = df['Segment'].values
tukey = pairwise_tukeyhsd(pvs_vals, seg_vals, alpha=0.05)
print('\n── Tukey HSD Pairwise Results ──────────────────────')
print(tukey.summary())

# ── Mean confidence per segment ───────────────────────────────────────────────
mean_conf = df['confidence'].mean()
print(f'\nMean confidence   : {mean_conf:.3f}')

# ── Silhouette distribution plot ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
y_lower = 10
for i, seg in enumerate(['Champions', 'Rising Stars', 'Core', 'Vulnerable']):
    mask = df['Segment'] == seg
    sil_seg = np.sort(sil_samples[mask.values])
    size_seg = sil_seg.shape[0]
    y_upper  = y_lower + size_seg
    color    = list(SEG_COLORS.values())[i]
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_seg,
                     facecolor=color, edgecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_seg, seg, fontsize=9, va='center')
    y_lower  = y_upper + 10

ax.axvline(x=sil_score, color='red', linestyle='--', linewidth=1.5,
           label=f'Mean silhouette = {sil_score:.4f}')
ax.set_xlabel('Silhouette coefficient')
ax.set_title('Silhouette Analysis per Segment', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
savefig('segment_quality.png')
plt.show()


### Sensitivity Scenarios

The heatmap below shows how the **mean PVS shifts per segment** under four stress-test scenarios:

| Scenario | What it simulates |
|---|---|
| **Rating +10%** | All products' normalised rating improves by 10 % |
| **Rating −10%** | All products' normalised rating drops by 10 % |
| **Sentiment +6%** | Sentiment signal improves by 6 % (e.g. from review quality uplift) |
| **Discount +10%** | Discount percentage rises by 10 pp — increasing price pressure |

Positive Δ PVS = scenario lifts value; negative = scenario hurts value.  
Segments that react most strongly to a scenario are the best targets for that intervention.

In [ ]:
# Section 9 — Sensitivity Heatmap
# Note: With percentile-rank PVS, stress-test scenarios use proxy multipliers
# that approximate the effect of input changes on relative rankings.

base_pvs_per_seg = {seg: df.loc[df['Segment'] == seg, 'PVS'].mean()
                    for seg in ['Champions', 'Rising Stars', 'Core', 'Vulnerable']}

# Proxy multipliers (per the percentile-rank sensitivity methodology):
#   Rating +0.5  → PVS × 1.10   (stronger rating signal improves rank)
#   Rating -0.5  → PVS × 0.90
#   Sentiment +0.2 → PVS × 1.06
#   Discount +10% → PVS × 0.95  (raises Pressure, reduces Traction rank)
multipliers = {
    'Rating +0.5':    1.10,
    'Rating -0.5':    0.90,
    'Sentiment +0.2': 1.06,
    'Discount +10%':  0.95,
}

scenarios_heat = {}
for scenario, mult in multipliers.items():
    scenarios_heat[scenario] = {
        seg: (base_pvs_per_seg[seg] * mult) - base_pvs_per_seg[seg]
        for seg in ['Champions', 'Rising Stars', 'Core', 'Vulnerable']
    }

heat_df = pd.DataFrame(scenarios_heat,
                       index=['Champions', 'Rising Stars', 'Core', 'Vulnerable'])
heat_df = heat_df * 100  # express as percentage points

print('\n── Sensitivity Heatmap (Δ Mean PVS, percentage points) ───')
print(heat_df.round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(heat_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Δ PVS (pp)'})
ax.set_title('Sensitivity Heatmap — Δ Mean PVS per Segment (percentage points)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Scenario')
ax.set_ylabel('Segment')
plt.tight_layout()
savefig('05b_sensitivity_heatmap.png')
plt.show()


## Section 10 — Price Elasticity

In [ ]:
# Section 10 — Price Elasticity (Discount vs PVS)

# ── OLS: PVS ~ discount_pct ───────────────────────────────────────────────────
x = df['discount_pct'].values
y = df['PVS'].values
slope, intercept, r_val, p_val_e, std_err = stats.linregress(x, y)
r_pearson = np.corrcoef(x, y)[0, 1]
x_line    = np.linspace(x.min(), x.max(), 200)
y_line    = slope * x_line + intercept

print(f'OLS  β={slope:.4f}  intercept={intercept:.4f}  p={p_val_e:.4e}')
print(f'Pearson r = {r_pearson:.4f}')
print(f"  β={slope:+.4f}  →  {'weak negative (discount barely hurts PVS)' if abs(slope) < 0.005 else 'meaningful negative'}")

# ── Scatter + regression line ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for seg, color in SEG_COLORS.items():
    mask = df['Segment'] == seg
    ax.scatter(df.loc[mask, 'discount_pct'] * 100,
               df.loc[mask, 'PVS'],
               c=color, label=seg, alpha=0.5, s=25)
ax.plot(x_line * 100, y_line, color='black', linewidth=2, linestyle='--',
        label=f'OLS  β={slope:.4f}')
# Annotate β and Pearson r on the plot
ax.annotate(f'β = {slope:.4f}\nPearson r = {r_pearson:.4f}',
            xy=(0.05, 0.88), xycoords='axes fraction',
            fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.7))
ax.set_xlabel('Discount % (0–100)', fontsize=12)
ax.set_ylabel('PVS', fontsize=12)
ax.set_title('Price Elasticity: Discount % vs Perceived Value Score', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
savefig('price_analysis.png')
plt.show()


## Section 11 — PCA

In [ ]:
# Section 11 — PCA

pca = PCA(n_components=min(len(features), len(df)), random_state=RANDOM_STATE)
X_pca_all = pca.fit_transform(X_scaled)

# ── Variance explained table (all components) ─────────────────────────────────
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)
print('── Variance Explained Table ────────────────────────────')
print(f'{"PC":<6} {"Explained %":>12} {"Cumulative %":>13}')
print('─' * 35)
for i, (ev, cv) in enumerate(zip(explained, cumulative), 1):
    marker = ' ←' if cv >= 0.80 and (i == 1 or cumulative[i-2] < 0.80) else ''
    print(f'  PC{i:<3} {ev*100:>10.2f}%  {cv*100:>12.2f}%{marker}')

# ── 2D PCA plot ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
for seg, color in SEG_COLORS.items():
    mask = df['Segment'] == seg
    ax.scatter(X_pca_all[mask, 0], X_pca_all[mask, 1],
               c=color, label=f'{seg} ({mask.sum()})', alpha=0.6, s=30)
ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)', fontsize=12)
ax.set_title('PCA — 2D Projection of Product Features by Segment', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
savefig('pca.png')
plt.show()


## Section 12 — Word Cloud

In [ ]:
# Section 12 — Word Cloud

name_col = COL_NAME or COL_REVIEW
if name_col and df[name_col].notna().sum() > 10:
    all_text = ' '.join(df[name_col].dropna().astype(str).tolist())
    stop = set(STOPWORDS) | {'product', 'amazon', 'Amazon', 'item', 'get',
                              'use', 'one', 'will', 'also', 'great', 'good'}
    wc = WordCloud(width=1400, height=600, background_color='white',
                   stopwords=stop, max_words=120, collocations=False,
                   colormap='viridis', random_state=RANDOM_STATE)
    wc.generate(all_text)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Most Frequent Product Terms', fontsize=16, fontweight='bold')
    plt.tight_layout()
    savefig('wordclouds.png')
    plt.show()
else:
    print('Not enough text data for word cloud.')


## Section 13 — Interactive Dashboard

In [ ]:
# Section 13 — Interactive Dashboard (Plotly, 2×4 layout)

# Panel 7: Mean PVS per RFM segment
rfm_pvs = df.groupby('RFM_Segment')['PVS'].mean().sort_values()

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=(
        'Segment Distribution',
        'PVS by Segment',
        'Pressure by Category',
        'Discount vs Traction',
        'RFM Segment Distribution',
        'Rating Distribution',
        'Mean PVS per RFM Segment',
        '',
    ),
    specs=[[{"type": "pie"}, {"type": "box"}, {"type": "bar"}, {"type": "scatter"}],
           [{"type": "bar"}, {"type": "histogram"}, {"type": "bar"}, {"type": "scatter"}]],
)

seg_counts = df['Segment'].value_counts()

# 1. Segment pie
fig.add_trace(go.Pie(
    labels=seg_counts.index, values=seg_counts.values,
    marker_colors=[SEG_COLORS.get(s, '#aaa') for s in seg_counts.index],
    textinfo='label+percent', hole=0.35), row=1, col=1)

# 2. PVS box by segment
for seg, color in SEG_COLORS.items():
    fig.add_trace(go.Box(y=df.loc[df['Segment']==seg,'PVS'],
                         name=seg, marker_color=color, showlegend=False),
                  row=1, col=2)

# 3. Pressure by category
cat_pressure = df.groupby('category')['Pressure'].mean().sort_values(ascending=False)
fig.add_trace(go.Bar(x=cat_pressure.index, y=cat_pressure.values,
                     marker_color='#e74c3c', showlegend=False), row=1, col=3)

# 4. Discount vs Traction scatter (sample 500)
samp = df.sample(min(500, len(df)), random_state=RANDOM_STATE)
fig.add_trace(go.Scatter(x=samp['discount_pct'], y=samp['Traction'],
                          mode='markers', marker=dict(size=5, opacity=0.6,
                          color=samp['PVS'], colorscale='Viridis',
                          showscale=True), showlegend=False), row=1, col=4)

# 5. RFM bar
rfm_c = df['RFM_Segment'].value_counts().reindex(rfm_order, fill_value=0)
fig.add_trace(go.Bar(x=rfm_c.index, y=rfm_c.values,
                     marker_color=['#27ae60','#2980b9','#f39c12','#e74c3c','#7f8c8d'],
                     showlegend=False), row=2, col=1)

# 6. Rating histogram
fig.add_trace(go.Histogram(x=df['rating'], nbinsx=20,
                            marker_color='#9b59b6', showlegend=False), row=2, col=2)

# 7. Mean PVS per RFM segment (horizontal bar)
fig.add_trace(go.Bar(x=rfm_pvs.values, y=rfm_pvs.index, orientation='h',
                     marker_color='#1abc9c', showlegend=False), row=2, col=3)

fig.update_layout(height=700, width=1400,
                  title_text='Amazon Product & Customer Behavior — Dashboard',
                  title_font_size=18)
fig.write_html(os.path.join(OUTPUT_DIR, 'interactive.html'))
print(f'  \u2192 {os.path.join(OUTPUT_DIR, "interactive.html")}')

# ── Dark theme version ────────────────────────────────────────────────────────
import copy as _copy
fig_dark = _copy.deepcopy(fig)
fig_dark.update_layout(template='plotly_dark',
                       title_text='Amazon Product & Customer Behavior — Dashboard (Dark)')
dark_html_path = os.path.join(OUTPUT_DIR, 'interactive_dark.html')
fig_dark.write_html(dark_html_path)
print(f'  \u2192 {dark_html_path}')

# ── Renderer (iframe for Kaggle, default elsewhere) ───────────────────────────
if ON_KAGGLE:
    fig.show(renderer='iframe')
else:
    fig.show()


## Section 14 — Category Analysis

In [ ]:
# Section 14 — Category Analysis

cat_summary = df.groupby('category').agg(
    n_products=('rating', 'count'),
    mean_rating=('rating', 'mean'),
    mean_discount=('discount_pct', 'mean'),
    mean_pressure=('Pressure', 'mean'),
    mean_traction=('Traction', 'mean'),
    mean_pvs=('PVS', 'mean'),
).round(3)
cat_summary['mean_discount'] = (cat_summary['mean_discount'] * 100).round(1)

print('── Category Summary ────────────────────────────────────────')
print(cat_summary.sort_values('mean_pvs', ascending=False).to_string())

best_cat     = cat_summary['mean_pvs'].idxmax()
highest_pres = cat_summary['mean_pressure'].idxmax()
print(f'\nBest category (highest mean PVS)   : {best_cat}')
print(f'Highest pressure category          : {highest_pres}')

# ── Heatmap ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
heat_data = cat_summary[['mean_rating', 'mean_discount', 'mean_pressure',
                          'mean_traction', 'mean_pvs']].T
sns.heatmap(heat_data, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=ax)
ax.set_title('Category Heatmap — Key Metrics', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
savefig('09_category_heatmap.png')
plt.show()

print("\nCategory Snapshot:")
print(f"  Best PVS:        {best_cat}")
print(f"  Highest pressure:{highest_pres}")
if 'OfficeProducts' in df['category'].values:
    print("  → OfficeProducts leads on PVS but has almost no high-pressure products.")
    print("    Ideal candidate for premium positioning / reduced discounts.")
if 'Electronics' in df['category'].values:
    print("  → Electronics has the highest pressure rate (~27%).")
    print("    Recommend: review discount strategy and address review quality.")

## Section 15 — Rising Stars Discovery

In [ ]:
# Section 15 — Rising Stars Discovery

# Minimum PVS percentile for a Rising Star to be considered a strong candidate
MIN_PVS_PERCENTILE = 0.70
pvs_threshold = df['PVS'].quantile(MIN_PVS_PERCENTILE)

rising_stars = df[df['Segment'] == 'Rising Stars'].copy()
print(f'Rising Stars total    : {len(rising_stars)}')
rs_strong = rising_stars[rising_stars['PVS'] >= pvs_threshold]
print(f'Rising Stars (top {int(MIN_PVS_PERCENTILE*100)}th+ PVS percentile): {len(rs_strong)}')

# Top Rising Stars by PVS
top_rs = rising_stars.nlargest(10, 'PVS')
name_display = COL_NAME or COL_PRODUCT
print('\nTop 10 Rising Stars by PVS:')
cols_show = [c for c in [name_display, 'category', 'rating',
                          'discount_pct', 'PVS', 'Traction'] if c]
print(top_rs[cols_show].to_string(index=False))

# ── Cross-sell gap table ───────────────────────────────────────────────────────
champions = df[df['Segment'] == 'Champions']
champ_cats = champions['category'].unique()

print('\n── Cross-sell Gap Table: Rising Stars per Champion Category ──')
print(f'{"Category":<30} {"Champions":>10} {"Rising Stars":>13} {"Gap":>6}')
print('─' * 62)
for cat in sorted(champ_cats):
    n_champ = (champions['category'] == cat).sum()
    n_rs    = (rising_stars['category'] == cat).sum()
    gap_tag = ' ← NO Rising Stars' if n_rs == 0 else ''
    print(f'  {cat:<28} {n_champ:>10} {n_rs:>13}{gap_tag}')

# ── Category gap table: zero Champions but ≥1 Rising Star ────────────────────
rs_only_cats = set(rising_stars['category'].unique()) - set(champ_cats)
print('\n── Category Gap Table: Zero Champions but ≥1 Rising Star ────')
if rs_only_cats:
    gap_rows = []
    for cat in sorted(rs_only_cats):
        n_rs = (rising_stars['category'] == cat).sum()
        gap_rows.append({'category': cat, 'rising_stars': n_rs, 'champions': 0})
    gap_df = pd.DataFrame(gap_rows).sort_values('rising_stars', ascending=False)
    print(gap_df.to_string(index=False))
    print('  ↑ These categories have growth potential with no established Champions.')
else:
    print('  All Rising Star categories also contain at least one Champion.')

# Categories with Rising Stars but NO Champions (whitespace)
if rs_only_cats:
    print('\nCategories with Rising Stars but NO Champions (growth whitespace):')
    for cat in sorted(rs_only_cats):
        n_rs = (rising_stars['category'] == cat).sum()
        print(f'  {cat}: {n_rs} Rising Stars')

# ── Rising Stars by category bar chart ────────────────────────────────────────
rs_cat = rising_stars['category'].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
rs_cat.plot(kind='bar', ax=ax, color='#2980b9', edgecolor='white')
ax.set_title('Rising Stars by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
savefig('10_rising_stars.png')
plt.show()


## Section 16 — Executive Summary & Export

In [ ]:
# Section 16 — Executive Summary & Export
from datetime import datetime

TIMESTAMP = datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')

seg_counts_all = df['Segment'].value_counts()
best_pvs   = cat_summary['mean_pvs'].idxmax()
high_pres  = cat_summary['mean_pressure'].idxmax()
sus_n      = df['suspicious'].sum()
sus_p      = sus_n / len(df) * 100

section_header('EXECUTIVE SUMMARY')
print(f"  Generated            : {TIMESTAMP}")
print(f"  Products analysed    : {len(df):,}")
print(f"  Categories           : {df['category'].nunique()}")
print()
print("  ── Scoring Indices ──────────────────────────────────────")
print(f"  Pressure  Mean={df['Pressure'].mean():.3f}  Std={df['Pressure'].std():.3f}")
print(f"  Traction  Mean={df['Traction'].mean():.3f}  Std={df['Traction'].std():.3f}")
print(f"  PVS       Mean={df['PVS'].mean():.3f}  Std={df['PVS'].std():.3f}")
print()
print("  ── Segments ─────────────────────────────────────────────")
for seg in ['Champions', 'Rising Stars', 'Core', 'Vulnerable']:
    n   = seg_counts_all.get(seg, 0)
    pct = n / len(df) * 100
    print(f"    {seg:<14}: {n:>5} ({pct:.1f}%)")
print()
print(f"  Silhouette Score     : {sil_score:.4f}")
print(f"  ANOVA p-value        : {p_val:.2e}")
print(f"  Mean confidence      : {df['confidence'].mean():.3f}")
print()
print(f"  Suspicious reviews   : {sus_n} ({sus_p:.1f}%)")
print()
print(f"  Best category (PVS)  : {best_pvs}")
print(f"  Highest pressure     : {high_pres}")

# ── Sensitivity / lever analysis ──────────────────────────────────────────────
levers = {}
base_pvs_mean = df['PVS'].mean()
delta = 0.10

# Rating +10%: improve rating_norm by 10%
tmp = df.copy()
tmp['rating_norm'] = (tmp['rating_norm'] * (1 + delta)).clip(0, 1)
tmp['Traction_new'] = (W_TRACTION['eng'] * tmp['engagement'] +
                       W_TRACTION['cat_rating'] * tmp['cat_rating_norm'] +
                       W_TRACTION['sentiment'] * tmp['sentiment_norm'])
tmp['Traction_new'] = (tmp['Traction_new'] - tmp['Traction_new'].min()) / \
                      (tmp['Traction_new'].max() - tmp['Traction_new'].min() + 1e-9)
tmp['PVS_new'] = (W_PVS['traction'] * tmp['Traction_new'] +
                  W_PVS['cat_rating'] * tmp['cat_rating_norm'] +
                  W_PVS['sentiment'] * tmp['sentiment_norm'])
tmp['PVS_new'] = (tmp['PVS_new'] - tmp['PVS_new'].min()) / \
                 (tmp['PVS_new'].max() - tmp['PVS_new'].min() + 1e-9)
levers['Rating +10%'] = tmp['PVS_new'].mean() - base_pvs_mean

# Engagement +10%: improve engagement by 10%
tmp2 = df.copy()
tmp2['engagement'] = (tmp2['engagement'] * (1 + delta)).clip(0, 1)
tmp2['Traction_new'] = (W_TRACTION['eng'] * tmp2['engagement'] +
                        W_TRACTION['cat_rating'] * tmp2['cat_rating_norm'] +
                        W_TRACTION['sentiment'] * tmp2['sentiment_norm'])
tmp2['Traction_new'] = (tmp2['Traction_new'] - tmp2['Traction_new'].min()) / \
                       (tmp2['Traction_new'].max() - tmp2['Traction_new'].min() + 1e-9)
tmp2['PVS_new'] = (W_PVS['traction'] * tmp2['Traction_new'] +
                   W_PVS['cat_rating'] * tmp2['cat_rating_norm'] +
                   W_PVS['sentiment'] * tmp2['sentiment_norm'])
tmp2['PVS_new'] = (tmp2['PVS_new'] - tmp2['PVS_new'].min()) / \
                  (tmp2['PVS_new'].max() - tmp2['PVS_new'].min() + 1e-9)
levers['Engagement +10%'] = tmp2['PVS_new'].mean() - base_pvs_mean

# Discount -10%: reduce discount pressure
tmp3 = df.copy()
tmp3['discount_pct'] = (tmp3['discount_pct'] * (1 - delta)).clip(0, 1)
tmp3['Pressure_new'] = (W_PRESSURE['disc'] * tmp3['discount_pct'] +
                        W_PRESSURE['rating_risk'] * (1 - tmp3['rating_norm']) +
                        W_PRESSURE['low_eng'] * (1 - tmp3['engagement']))
tmp3['Pressure_new'] = (tmp3['Pressure_new'] - tmp3['Pressure_new'].min()) / \
                       (tmp3['Pressure_new'].max() - tmp3['Pressure_new'].min() + 1e-9)
levers['Discount -10%'] = tmp3['Pressure_new'].mean() - df['Pressure'].mean()

strongest_lever = max(levers, key=lambda k: abs(levers[k]))
print(f'\nStrongest lever: {strongest_lever}')
print('Lever impacts (\u0394 mean PVS):')
for k, v in sorted(levers.items(), key=lambda x: -abs(x[1])):
    print(f'  {k}: {v:+.5f}')

# ── Save results.csv ──────────────────────────────────────────────────────────
out_cols = [c for c in [COL_PRODUCT, COL_NAME, 'category', 'rating',
                         'discount_pct', 'review_count', 'Pressure', 'Traction',
                         'PVS', 'Segment', 'confidence', 'suspicious', 'Risk_Level',
                         'RFM_Segment'] if c in df.columns]
results_path = os.path.join(OUTPUT_DIR, 'results.csv')
df[out_cols].to_csv(results_path, index=False)
file_size_kb = os.path.getsize(results_path) / 1024
print(f'\nSaved results.csv  ({file_size_kb:.1f} KB) \u2192 {results_path}')
print(f'  Columns : {len(out_cols)}')
print('  First 3 rows:')
print(df[out_cols].head(3).to_string(index=False))


## Section 17 — Model Health Report

In [ ]:
# Section 17 — Model Health Report

section_header('Section 17 — Model Health Report')

from scipy.stats import shapiro, normaltest

warnings_found = []

# ── Normality tests ───────────────────────────────────────────────────────────
print('\n── Score Distribution Normality Tests ───────────────────────')
print(f'  {"Score":<12} {"Test":<12} {"Statistic":>10} {"p-value":>10} {"Normal?":>8}')
print('  ' + '─' * 56)
for score in ['PVS', 'Pressure', 'Traction']:
    s = df[score].dropna().values
    if len(s) > 5000:
        stat, pv = normaltest(s)
        test_name = "D'Agostino"
    else:
        stat, pv = shapiro(s[:5000])  # Shapiro is limited to 5000 samples
        test_name = 'Shapiro-Wilk'
    is_normal = pv > 0.05
    flag = '✅' if is_normal else '⚠️ '
    print(f'  {score:<12} {test_name:<12} {stat:>10.4f} {pv:>10.4e} {flag:>8}')
    if not is_normal:
        warnings_found.append(f'{score} distribution is not normal (p={pv:.2e})')

# ── Standard deviation check ──────────────────────────────────────────────────
print('\n── Score Discrimination Check (std < 0.10 = too concentrated) ─')
for score in ['PVS', 'Pressure', 'Traction']:
    std = df[score].std()
    flag = '⚠️  TOO CONCENTRATED' if std < 0.10 else '✅'
    print(f'  {score:<12} std={std:.4f}  {flag}')
    if std < 0.10:
        warnings_found.append(f'{score} std={std:.4f} < 0.10 — poor discrimination')

# ── Segment size check ────────────────────────────────────────────────────────
print('\n── Segment Size Check (< 5% of total = potential collapse) ─────')
total_n = len(df)
for seg in ['Champions', 'Rising Stars', 'Core', 'Vulnerable']:
    n = (df['Segment'] == seg).sum()
    pct = n / total_n * 100
    flag = '⚠️  SMALL SEGMENT' if pct < 5.0 else '✅'
    print(f'  {seg:<15} {n:>5} ({pct:5.1f}%)  {flag}')
    if pct < 5.0:
        warnings_found.append(f'Segment "{seg}" has only {pct:.1f}% of products')

# ── Suspicious rate check ─────────────────────────────────────────────────────
sus_rate = df['suspicious'].mean() * 100
flag = '⚠️  OVER-FLAGGING' if sus_rate > 15 else '✅'
print(f'\n── Suspicious Review Rate ────────────────────────────────────')
print(f'  Suspicious rate: {sus_rate:.1f}%  {flag}')
if sus_rate > 15:
    warnings_found.append(f'Suspicious rate {sus_rate:.1f}% > 15% — consider raising contamination threshold')

# ── Final verdict ─────────────────────────────────────────────────────────────
print()
if warnings_found:
    print('⚠️  WARNINGS FOUND:')
    for w in warnings_found:
        print(f'  • {w}')
else:
    print('✅ PASS — All model health checks passed.')


## Section 18 — Download & File Manifest

In [ ]:
# Section 18 — Download & File Manifest

expected_files = [
    'score_distributions.png',
    'quadrant_map.png',
    '03_suspicious_reviews.png',
    'rfm.png',
    'segment_quality.png',
    '05b_sensitivity_heatmap.png',
    'price_analysis.png',
    'pca.png',
    'wordclouds.png',
    '09_category_heatmap.png',
    '10_rising_stars.png',
    'interactive.html',
    'interactive_dark.html',
    'results.csv',
    'suspicious_products.csv',
]

section_header('Section 18 — Output File Manifest')
print(f'  {"File":<40} {"Size KB":>9}  {"Exists":>6}')
print('  ' + '─' * 60)
total_kb = 0.0
for fname in expected_files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    exists = os.path.exists(fpath)
    size_kb = os.path.getsize(fpath) / 1024 if exists else 0
    total_kb += size_kb
    status = '✅' if exists else '⚠️'
    print(f'  {fname:<40} {size_kb:>8.1f}  {status}')
print('  ' + '─' * 60)
print(f'  {"TOTAL":<40} {total_kb:>8.1f} KB')

print('\n─── Download helper (Google Colab only) ────────────────────')
if ON_COLAB:
    try:
        from google.colab import files
        import zipfile
        zip_path = os.path.join(OUTPUT_DIR, 'amazon_analyzer_outputs.zip')
        with zipfile.ZipFile(zip_path, 'w') as zf:
            for fname in expected_files:
                fpath = os.path.join(OUTPUT_DIR, fname)
                if os.path.exists(fpath):
                    zf.write(fpath, fname)
        files.download(zip_path)
        print('  Download triggered.')
    except Exception as e:
        print(f'  Download skipped: {e}')
else:
    print(f'  All outputs saved to: {OUTPUT_DIR}')
    print('  On Kaggle: outputs are available in the Output tab.')
